In [1]:
import math
import copy
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2)

| Size (sqft) | Number of Bedrooms  | Number of floors | Age of  Home | Price (1000s dollars)  |   
| ----------------| ------------------- |----------------- |--------------|-------------- |  
| 2104            | 5                   | 1                | 45           | 460           |  
| 1416            | 3                   | 2                | 40           | 232           |  
| 852             | 2                   | 1                | 35           | 178           |  

In [2]:
# load dataset
X_train = np.array([[2104, 5, 1, 45], [1416, 3, 2, 40], [852, 2, 1, 35]])
y_train = np.array([460, 232, 178])

print(f"X_train: {X_train}") # input features
print(f"y_train: {y_train}") # output targets

X_train: [[2104    5    1   45]
 [1416    3    2   40]
 [ 852    2    1   35]]
y_train: [460 232 178]


In [3]:
X_train.shape
# it will return (m,n) 
# m = no. of training examples, n = no. of input features

(3, 4)

In [4]:
b_init = 785.1811367994083 # a scalar
w_init = np.array([ 0.39133535, 18.75376741, -53.36032453, -26.42131618]) # vector

print(f"w_init shape: {w_init.shape}, w_init type: {type(w_init)}")
print(f"b_init type: {type(b_init)}")

w_init shape: (4,), w_init type: <class 'numpy.ndarray'>
b_init type: <class 'float'>


The model's prediction with multiple variables is given by the linear model:

$$ f_{\mathbf{w},b}(\mathbf{x}) =  w_0x_0 + w_1x_1 +... + w_{n-1}x_{n-1} + b \tag{1}$$
or in vector notation:
$$ f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b  \tag{2} $$ 
where $\cdot$ is a vector `dot product`

To demonstrate the dot product, we will implement prediction using (1) and (2).

In [5]:
def predict_single_loop(x, w, b):
    # x is the input features for one training example -> vector
    # w is the weight parameters of model -> vector
    # b is bias parameter -> scalar

    n = x.shape[0] # no. of features
    p = 0
    for i in range(n):
        p_i = x[i] * w[i]
        p += p_i
    p += b
    return p    

In [6]:
# get a row from our training data
x_vec = X_train[0,:] # 0th row and all columns
print(f"x_vec: {x_vec}")
print(f"x_vec shape: {x_vec.shape}")

x_vec: [2104    5    1   45]
x_vec shape: (4,)


In [7]:
# make a prediction
f_wb = predict_single_loop(x_vec, w_init, b_init)
print(f"f_wb: {f_wb}")
print(f"f_wb shape: {f_wb.shape}")

f_wb: 459.9999976194083
f_wb shape: ()


In [8]:
def predict(x, w, b):
    p = np.dot(x,w) + b
    return p

In [9]:
# now again
x_vec = X_train[0]
print(f"x_vec: {x_vec}")
print(f"x_vec shape: {x_vec.shape}\n")

f_wb = predict(x_vec, w_init, b_init)
print(f"f_wb: {f_wb}")
print(f"f_wb shape: {f_wb.shape}")

x_vec: [2104    5    1   45]
x_vec shape: (4,)

f_wb: 459.9999976194083
f_wb shape: ()


The equation for the cost function with multiple variables $J(\mathbf{w},b)$ is:
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})^2 \tag{3}$$ 
where:
$$ f_{\mathbf{w},b}(\mathbf{x}^{(i)}) = \mathbf{w} \cdot \mathbf{x}^{(i)} + b  \tag{4} $$ 


In [10]:
def compute_cost(X, y, w, b):
    m = X.shape[0] # no. of training examples
    cost_sum = 0
    for i in range(m):
        f_wb = np.dot(X[i], w) + b # our model prediction
        cost = (f_wb - y[i]) ** 2
        cost_sum += cost
    total_cost = cost_sum / (2 * m)
    return total_cost

In [11]:
# compute and display cost using our pre chosen optimal parameters

cost = compute_cost(X_train, y_train, w_init, b_init)
print(f"Cost at optimal w: {cost}")

Cost at optimal w: 1.5578904428966628e-12


Gradient descent for multiple variables:

$$\begin{align*} \text{repeat}&\text{ until convergence:} \; \lbrace \newline\;
& w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{5}  \; & \text{for j = 0..n-1}\newline
&b\ \ = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b}  \newline \rbrace
\end{align*}$$

where, n is the number of features, parameters $w_j$,  $b$, are updated simultaneously and where  

$$
\begin{align}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{6}  \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{7}
\end{align}
$$
* m is the number of training examples in the data set

    
*  $f_{\mathbf{w},b}(\mathbf{x}^{(i)})$ is the model's prediction, while $y^{(i)}$ is the target value

In [12]:
def compute_gradient(X, y, w, b):
    m,n = X.shape # no. of examples and no. of features

    dj_dw = np.zeros(n)
    dj_db = 0

    for i in range(m):
        err = (np.dot(X[i], w) + b) - y[i]
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err * X[i,j]
        dj_db = dj_db + err
    dj_dw = dj_dw / m
    dj_db = dj_db / m

    return dj_dw, dj_db    

In [13]:
# compute and display gradient
tmp_dj_dw, tmp_dj_db = compute_gradient(X_train, y_train, w_init, b_init)
print("Current gradient at current values for w and b: \n")
print(f"tmp_dj_dw: {tmp_dj_dw}")
print(f"tmp_df_db: {tmp_dj_db}")

Current gradient at current values for w and b: 

tmp_dj_dw: [-2.73e-03 -6.27e-06 -2.22e-06 -6.92e-05]
tmp_df_db: -1.6739251501955248e-06


In [14]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters, compute_cost, compute_gradient):
    
    j_history = [] # cost history while our parameters are changing
    w = copy.deepcopy(w_in)
    b = b_in
    
    for i in range(num_iters):

        dj_dw, dj_db = compute_gradient(X, y, w, b)

        w = w - alpha * dj_dw
        b = b - alpha * dj_db

        j_history.append(compute_cost(X, y, w, b))


        if i % math.ceil(num_iters/10) == 0:
            print(f"Iteration {i:4d}: Cost {j_history[-1]:8.2f}   ")

    return w, b, j_history

In [15]:
initial_w = np.zeros_like(w_init)
initial_b = 0.

iterations = 1000
alpha = 5.0e-7

final_w, final_b, j_hist = gradient_descent(X_train, y_train, initial_w, initial_b, alpha, iterations, compute_cost, compute_gradient)

print(f"\n\nb, w found by gradient descent: {final_b:.2f}, {final_w}\n")

m,_ = X_train.shape

for i in range(m):
    print(f"prediction: {np.dot(X_train[i],final_w) + final_b}, target value: {y_train[i]}")

Iteration    0: Cost  2529.46   
Iteration  100: Cost   695.99   
Iteration  200: Cost   694.92   
Iteration  300: Cost   693.86   
Iteration  400: Cost   692.81   
Iteration  500: Cost   691.77   
Iteration  600: Cost   690.73   
Iteration  700: Cost   689.71   
Iteration  800: Cost   688.70   
Iteration  900: Cost   687.69   


b, w found by gradient descent: -0.00, [ 0.2   0.   -0.01 -0.07]

prediction: 426.18530497189204, target value: 460
prediction: 286.1674720078562, target value: 232
prediction: 171.46763087132317, target value: 178
